In [1]:
from langchain_community.document_loaders import PyPDFLoader, PyPDFDirectoryLoader
import copy
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from dotenv import load_dotenv
%pip install langchain-classic
%pip install rank_bm25
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.document_compressors.flashrank_rerank import FlashrankRerank
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_classic.storage import LocalFileStore
from langchain_classic.storage._lc_store import create_kv_docstore
from langchain_core.runnables import RunnableLambda


C:\Users\hp\anaconda3\envs\langchain-environment\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
path_to_folder = "./data-reg"
loader = PyPDFDirectoryLoader(path_to_folder)
documents = loader.load()
print(f"Total pages loaded: {len(documents)}")

Total pages loaded: 1237


In [3]:
for i, doc in enumerate(documents[:2]):
    print(f"--- DOCUMENT {i+1} ---")
    
    source = doc.metadata.get('source', 'Unknown Source')
    page = doc.metadata.get('page', 'Unknown Page')
    print(f"SOURCE: {source} (Page {page})")
    
    content_snippet = doc.page_content[:500].replace('\n', ' ')
    print(f"CONTENT: {content_snippet}...\n")

--- DOCUMENT 1 ---
SOURCE: data-reg\doc1.pdf (Page 0)
CONTENT: 1             Ref: CBN/MPC/COM/154/297  Attention: News Editors/Gentlemen of the Press    MONETARY POLICY RATE RAISED BY 50 BASIS POINTS TO 27.25 PER CENT TO 26.75  PER CENT    The Monetary Policy Committee (MPC) of the Central Bank of Nigeria (CBN)  held its 297th meeting on the 23rd and 24th of September 2024 to review recent  economic and financial developments as well as assess risks to the outlook.  Eleven of the twelve members of the Committee were in attendance.     Decisions of the MPC  ...

--- DOCUMENT 2 ---
SOURCE: data-reg\doc1.pdf (Page 1)
CONTENT: 2    Considerations  The Committee noted the moderation in headline inflation year-on-year in July  and August 2024. In addition, the MPC noted the relative stability and  convergence in the exchange rate across the various market segments,  resulting from the Bank’s tight monetary policy stance. This is expected to  improve confidence which will enable economic agen

In [4]:
parent_rag_path = r"C:\Users\hp\Central Insight"

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore_parent = Chroma(persist_directory=parent_rag_path, embedding_function=embeddings)

fs = LocalFileStore(r"C:\Users\hp\Central Insight\docstore")
store = create_kv_docstore(fs)

parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore_parent,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)


parent_retriever.search_type = "mmr"
parent_retriever.search_kwargs = {
    'k': 15,
    'fetch_k': 50,
    'lambda_mult': 0.5,
   
}


if not list(store.yield_keys()):
    batch_size = 5
    print(f"Starting indexing for {len(documents)} documents...")
    for i in range(0, len(documents), batch_size):
        batch = documents[i : i + batch_size]
        parent_retriever.add_documents(batch)
    print(f"Indexing complete. Database saved to: {parent_rag_path}")
else:
    print("Store already populated — skipping indexing.")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2197.49it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting indexing for 1237 documents...
Indexing complete. Database saved to: C:\Users\hp\Central Insight


In [5]:
child_docs_for_bm25 = child_splitter.split_documents(documents)
bm25_retriever = BM25Retriever.from_documents(child_docs_for_bm25)
bm25_retriever.k = 15


ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, parent_retriever],
    weights=[0.5, 0.5]
)


print("Done with ensemble")

Done with ensemble


In [6]:
compressor = FlashrankRerank(model="ms-marco-MiniLM-L-12-v2", top_n=10)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, 
    base_retriever=ensemble_retriever
)

print("Compressor Pipeline Rebuilt")

Compressor Pipeline Rebuilt


In [7]:
load_dotenv(r"C:\Users\hp\Central Insight\.env")

llm = ChatGroq(model_name="llama-3.3-70b-versatile", max_tokens=1200, temperature=0, model_kwargs={"seed": 365})




In [8]:
PROMPT_TEMPLATE = """
You are a Senior Regulatory Analyst at NaijaReg, specializing in Central Bank of Nigeria (CBN) and Monetary Policy Committee (MPC) directives.
Your goal is to provide precise, factual, and professional summaries of regulatory changes based ONLY on the provided context.

### GUIDELINES:
1. **Source Only:** Answer using ONLY the provided context. If the answer is not in the context, say: "I am sorry, but the provided documents do not contain information regarding this specific query."
2. **Numeric Precision:** Quote all rates (MPR, CRR, Liquidity Ratio) and basis points exactly as they appear.
3. **Structured Format:** Use bullet points for specific policy changes.
4. **Tone:** Maintain a formal, analytical, and objective tone.
5. **Stay On Topic:** Answer ONLY what the question asks. Do NOT volunteer additional policy changes, 
   decisions, or data that were not specifically requested.

### CONTEXT:
{context}

### QUESTION:
{question}

### RESPONSE FORMAT:
- Start with a clear 1-2 sentence summary that directly answers the question.
- Use a "Policy Changes" section ONLY for details directly relevant to the question.
- End with a "Sources" section citing the document name and page number exactly as shown in the context headers.
"""

In [9]:
prompt = PromptTemplate.from_template(PROMPT_TEMPLATE)

In [10]:
def format_docs(docs):
    formatted = []
    for doc in docs:
        source = doc.metadata.get('source', 'Unknown Source')
        page   = doc.metadata.get('page', 'Unknown Page')
        formatted.append(f"[Source: {source}, Page: {page}]\n{doc.page_content}")
    return "\n\n".join(formatted)



refined_chain = (
    {
        "context": compression_retriever | format_docs, 
        "question": RunnablePassthrough()
    } 
    | prompt 
    | llm 
    | StrOutputParser()
)



In [11]:
REWRITE_TEMPLATE = """
You are an expert in Central Bank of Nigeria regulatory documents.
Rewrite the following user question into precise regulatory language 
that would match how CBN/MPC documents are written.
Return ONLY the rewritten question, nothing else.

User question: {question}
Rewritten question:
"""

rewrite_prompt = PromptTemplate.from_template(REWRITE_TEMPLATE)
rewrite_chain = rewrite_prompt | llm | StrOutputParser()


new_chain = (
    {
        "context": RunnableLambda(lambda x: rewrite_chain.invoke({"question": x})) 
                   | compression_retriever 
                   | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [15]:
result = new_chain.invoke("When was the first MPC meeting in 2023?")

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [16]:
print(result)

The first MPC meeting in 2023 was held on January 2023, although the exact dates are not specified in the provided context. This meeting is mentioned as a reference point for economic developments and policy decisions.

### Policy Changes
* The meeting is noted to have occurred in January 2023, with indications of continued weak economic momentum from 2022 into 2023.
* There were strong fears of a global recession in 2023, with about one-third of the global economy facing recession.

### Sources
[Source: data-reg\doc17.pdf, Page: 29]
